## Simulating Sample Paths
### ACT460, Fall 2025

In this tutorial, we study different ways to simulate geometric Brownian motion.

In [17]:
# load Python packages
import numpy as np  # for numerical python libraries
import matplotlib.pyplot as plt  # for plotting

## Exact simulation of GBMs

In this example, we simulate exactly sample paths from a geometric Brownian motion (GBM) and inspect the effects of the different parameters. Recall that GBM has SDE
$$ dS_t = \mu S_t dt + \sigma S_t dW_t$$

In [18]:
def simulate_gbm_exact(S0, mu, sigma, T, Ndt, Nsims=10):
    """
    Simulate two-dimensional Geometric Brownian Motion with correlation rho.

    Parameters:
    S0 : float
        Initial value of the process.
    mu : float
        Drift coefficient of the process.
    sigma : float
        Volatility coefficient of the process.
    T : float
        Total time period to simulate.
    Ndt : float
        Number of time steps to take.

    Returns:
    prices : ndarray
        Simulated process over time.
    """
        
    # create time increment
    dt = T / Ndt
    
    # set the different times as a vector
    time = np.linspace(0, T, Ndt+1)
    
    # initialize the matrix for GBMs
    S = np.zeros((Ndt+1,Nsims))
    
    # initial value of the process
    S[0,:] = S0
    
    # recursively simulate the process ( S[idx,:] is (1 , Nsims) )
    for idx in range(Ndt):
        Z = np.random.randn(Nsims)
        # or Z = np.random.normal(0.0, 1.0, size=(1,Nsims))
        S[idx+1,:] = S[idx,:] * np.exp( (mu - (sigma**2)/2) * dt
                                      + sigma * np.sqrt(dt) * Z)
    
    return time, S

In [19]:


# Model parameters
S0 = 100
K = 95
r = 0.03
q = 0.02
sigma = 0.25
T = 0.5
m = 50

# Simulation settings
mu = r - q
Ndt = m
Nsims = 10000

# Random seed for reproducibility
np.random.seed(460)

time, S = simulate_gbm_exact(S0, mu, sigma, T, Ndt, Nsims)

S_monitor = S[1:, :]

log_S_monitor = np.log(S_monitor)
mean_log_S = np.mean(log_S_monitor, axis=0)
S_G = np.exp(mean_log_S)
print(S_G.shape)
payoff = np.maximum(K-S_G, 0)
discount = np.exp(-r * T)
P_G = np.mean(discount * payoff)
P_G_sd = np.std(discount * payoff, ddof=1) / np.sqrt(Nsims)
print(P_G)
P_G_sd = np.std(discount * payoff, ddof=1) / np.sqrt(Nsims)
print(P_G_sd)

(10000,)
1.9631263677358652
0.038581418327750236


In [20]:
from scipy.stats import norm

mu_G = np.log(100) + (r - q - sigma**2/2) * (T*(m+1))/(2*m)

v_G = sigma**2*T*(m+1)*(2*m+1)/(6*m**2)

d = (np.log(K) - mu_G)/np.sqrt(v_G)

A = K*norm.cdf(d)

B = np.exp(mu_G + v_G/2)*norm.cdf(d - np.sqrt(v_G))

P_G_a = np.exp(-r*T)*(A-B)

print(P_G_a)

1.9825017052583809
